In [ ]:
import pandas as pd
import plotly.express as px

# same palette as Alina's charts so all four plots match
SURFACE = "#fcfcfb"
INK     = "#0b0b0b"
INK_2   = "#52514e"
MUTED   = "#898781"
GRID    = "#e1e0d9"
RED     = "#eb345c"   # Alina's "counties" color

def style(fig, title, subtitle, source="Guttmacher Institute, 2020."):
    """Bold left-aligned headline + grey subtitle + source line on an off-white background."""
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><span style='font-size:14px;color:{INK_2}'>{subtitle}</span>",
                   x=0.03, xanchor="left", y=0.95, font=dict(size=22, color=INK)),
        font=dict(family="DejaVu Sans, Arial", color=INK_2),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    )
    fig._source = source
    return fig

def add_source(fig):
    """Source line in the bottom margin (call after setting the margins)."""
    fig.add_annotation(text=fig._source, x=0, y=0, xref="paper", yref="paper",
                       xanchor="left", yanchor="bottom", yshift=-fig.layout.margin.b + 8,
                       showarrow=False, font=dict(size=11, color=MUTED))

In [6]:
df= pd.read_csv("../abortion_clean.csv")

df.head()

,state,state_abbr,pct_chg_abortion_rate_2017_2020,pct_chg_clinics_2017_2020,pct_chg_providers_2014_2017,pct_us_abortions_occurrence_2020,pct_counties_no_provider_2014,pct_counties_no_clinic_2020,pct_traveled_out_of_state_2020,pct_women_no_clinic_2020,...,rate_residence_2020,abortions_occurrence_2020,abortions_residence_2020,federally_funded_abortions_2010,state_funded_abortions_2010,public_exp_federal_2015_k,public_exp_state_2015_k,total_publicly_funded_abortions_2010,total_public_exp_2015_k,women_15_44_2020
0,Alabama,AL,-6.0,0.0,-22.0,0.6,91.0,93.0,47.0,59.0,...,9.5,5700.0,9060.0,9.0,0.0,22.0,0.0,9.0,22.0,953684.0
1,Alaska,AK,0.0,0.0,-25.0,0.1,79.0,87.0,7.0,33.0,...,9.2,1240.0,1320.0,0.0,835.0,0.0,216.0,835.0,216.0,143478.0
2,Arizona,AZ,1.0,0.0,-8.0,1.4,80.0,80.0,6.0,18.0,...,9.7,13320.0,13820.0,1.0,13.0,28.0,11.0,14.0,40.0,1424742.0
3,Arkansas,AR,2.0,-33.0,0.0,0.3,97.0,99.0,37.0,86.0,...,7.8,3250.0,4510.0,0.0,0.0,0.0,0.0,0.0,0.0,578205.0
4,California,CA,17.0,7.0,-18.0,16.6,24.0,38.0,0.0,3.0,...,19.0,154060.0,152400.0,0.0,88466.0,0.0,32613.0,88466.0,32613.0,8021053.0


In [7]:
fig = px.choropleth(df, locations="state_abbr", locationmode="USA-states", scope="usa",
                    color="pct_counties_no_clinic_2020", range_color=(0,100),
                    color_continuous_scale=["#fde4e9", RED, "#7a0f26"],
                    hover_name="state")
style(fig, "Most of America has no abortion clinic",
      "In the average state, 78% of counties have no known abortion clinic (2020)")
fig.update_traces(marker_line_color="white", marker_line_width=1,
                  hovertemplate="<b>%{hovertext}</b><br>%{z}% of counties have no clinic<extra></extra>")
fig.update_geos(showlakes=False, bgcolor=SURFACE, landcolor=SURFACE)
fig.update_coloraxes(colorbar=dict(title="", ticksuffix="%", orientation="h", thickness=12, len=0.5,
                                   x=0.5, y=-0.04, yanchor="top", outlinewidth=0, tickfont=dict(color=MUTED)))
fig.update_layout(width=900, height=640, margin=dict(l=20, r=20, t=100, b=80))
add_source(fig)
fig.write_image("pro_plot1_map.png", scale=2)
fig

In [8]:
# only the 17 states that lost clinics (cherry-picked)
lost = df[df.pct_chg_clinics_2017_2020 < 0]
clinics_2020 = lost.clinics_2020.sum()
clinics_2017 = (lost.clinics_2020 - lost.chg_clinics_2017_2020).sum()

totals = pd.DataFrame({"year": ["2017", "2020"], "clinics": [clinics_2017, clinics_2020]})
fig2 = px.bar(totals, x="year", y="clinics", text="clinics",
              color="year", color_discrete_map={"2017": "#c3c2b7", "2020": RED})  # grey "before", red "now"
style(fig2, "Abortion clinics are disappearing across 17 states",
      "Number of abortion clinics, 2017 vs 2020")
fig2.update_traces(texttemplate="<b>%{y:.0f}</b>", textposition="outside",
                   textfont=dict(size=20, color=INK), width=0.55)
fig2.update_xaxes(title="", tickfont=dict(size=15, color=INK), showline=True, linecolor="#c3c2b7")
fig2.update_yaxes(range=[340, 405],  # truncated axis: an 11% drop looks like a collapse
                  title="", gridcolor=GRID, tickfont=dict(color=MUTED), zeroline=False)
fig2.update_layout(width=700, height=560, margin=dict(l=50, r=30, t=110, b=50), showlegend=False)
add_source(fig2)
fig2.write_image("pro_plot2_clinics.png", scale=2)
fig2